# Capstone: Content Opportunity Scoring

**Content Retention lane**

This notebook mirrors the public research paper. It is structured as a short, reproducible walkthrough of the decision, data contract, model evidence, limitations, and editorial playbook.

## 1. Question

Can historical search visibility metrics identify pages showing search traffic decay so editors can prioritize organic-search refreshes rather than guess?

The output is decision-support: a probability-ranked queue for human review. It is not an automated publisher and does not claim causal recovery.

In [1]:
from pathlib import Path
import json
import pandas as pd

# Locate the repository from either a local checkout or a notebook runtime.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / ".git").exists():
    repo_root = repo_root.parent
metrics_path = repo_root / "work" / "outputs" / "capstone_metrics.json"
metrics = json.loads(metrics_path.read_text())

print("Lane:", metrics["lane"])
print("Question: rank observed search-decay symptoms for editorial review")
print("Metrics receipt:", metrics_path)

Lane: Content Opportunity Scoring (Content Retention)
Question: rank observed search-decay symptoms for editorial review
Metrics receipt: /Users/ibrahimyaser/Documents/CS Study/flyrank-ibrahim/work/outputs/capstone_metrics.json


## 2. Data

The source is the `fact_content_daily_performance` table from the FlyRank warehouse, queried with DuckDB on Hugging Face. Its grain is one row per report date, pseudonymous client, and pseudonymous content item.

The time-aware design uses `month=2026-03` for training and the June 2026 final-month `_sample` exclusively for testing. The target is `needs_refresh = (clicks = 0 AND impressions > 0)`.

The requested feature contract is `impressions`, `position`, and `is_weekend`. Mathematically derived rates such as `ctr`, future metrics, label-derived fields, and pseudonymous IDs are excluded from that contract to reduce leakage risk.

In [2]:
requested_features = metrics["requested_features"]
forbidden_features = {"ctr", "clicks", "trend_direction", "trend_pct", "content_hash_id", "client_hash_id"}
assert requested_features == ["impressions", "position", "is_weekend"]
assert not forbidden_features.intersection(requested_features)
assert metrics["training_split"] == "month=2026-03"
assert "June 2026" in metrics["test_split"]
print("Requested features:", requested_features)
print("Training:", metrics["training_split"])
print("Testing:", metrics["test_split"])
print("Leakage exclusions verified:", sorted(forbidden_features))

Requested features: ['impressions', 'position', 'is_weekend']
Training: month=2026-03
Testing: June 2026 final-month sample (_sample)
Leakage exclusions verified: ['clicks', 'client_hash_id', 'content_hash_id', 'ctr', 'trend_direction', 'trend_pct']


## 3. Methodology

The Random Forest classifier uses 100 trees, maximum depth 8, balanced class weights, and seed 42. A forest can represent non-linear interactions among visibility, rank, and calendar context.

The transparent Week 4 baseline flags `position <= 10 AND impressions > 1000 AND ctr < 0.02`. It is useful because an editor can inspect every condition, but the fixed thresholds cannot adapt to combinations of signals.

**Evidence boundary.** The archived `w05_model.ipynb` execution used `impressions`, `ctr`, and `is_weekend` because its query could not resolve `position`. The receipt preserves that fact. The requested position-based feature contract above is the configuration to rerun before treating the archived scores as a final benchmark.

In [3]:
model_config = metrics["model"]
print("Model configuration:")
print(json.dumps(model_config, indent=2))
print("Target:", metrics["label"])
print("Archived executed w05 features:", metrics["executed_w05_features"])
assert model_config["random_state"] == 42
assert model_config["n_estimators"] == 100
assert model_config["max_depth"] == 8

Model configuration:
{
  "name": "Random Forest classifier",
  "n_estimators": 100,
  "max_depth": 8,
  "class_weight": "balanced",
  "random_state": 42
}
Target: needs_refresh = (clicks = 0 AND impressions > 0)
Archived executed w05 features: ['impressions', 'ctr', 'is_weekend']


## 4. Results: model versus baseline

These are the exact rendered Precision, Recall, and Accuracy values from `w05_model.ipynb`, comparing both methods on the same June 2026 split shown in that notebook. The baseline and model values are kept as a committed receipt so the paper's numbers are inspectable without committing warehouse data.

In [4]:
archived = metrics["archived_w05_metrics"]
results = pd.DataFrame({
    "Metric": ["Precision", "Recall", "Accuracy"],
    "Week 4 fixed baseline": [archived["baseline"][key] for key in ["precision", "recall", "accuracy"]],
    "Random Forest": [archived["random_forest"][key] for key in ["precision", "recall", "accuracy"]],
})
print(results.to_string(index=False, formatters={
    "Week 4 fixed baseline": "{:.2%}".format,
    "Random Forest": "{:.2%}".format,
}))
assert archived["baseline"] == {"precision": 0.1289, "recall": 0.0044, "accuracy": 0.6189}
assert archived["random_forest"] == {"precision": 0.9997, "recall": 1.0, "accuracy": 0.9999}

   Metric Week 4 fixed baseline Random Forest
Precision                12.89%        99.97%
   Recall                 0.44%       100.00%
 Accuracy                61.89%        99.99%


The archived Random Forest measured 99.97% precision, 100.00% recall, and 99.99% accuracy, compared with 12.89%, 0.44%, and 61.89% for the fixed baseline. The directional interpretation is that the model learned non-linear relationships in the available metrics and reduced false positives relative to the rigid rule. These scores describe proxy-label classification, not guaranteed traffic recovery.

## 5. Validation and leakage audit

The validation notebook measured 92.04% precision on a random split and 95.67% on a March-to-April time-aware split. The capstone's primary design remains temporal: March is training data and June is the reserved final-month test window.

The leakage audit found no feature with suspicious correlation above 0.8 with the target in the inspected time-aware slice. This does not prove that every possible leakage path is absent; it documents the check that was performed.

In [5]:
audit = metrics["validation_audit"]
validation = pd.DataFrame({
    "Split": ["Random split", "March -> April time-aware split"],
    "Precision": [audit["random_split_precision"], audit["time_aware_precision"]],
})
print(validation.to_string(index=False, formatters={"Precision": "{:.2%}".format}))
assert audit["time_aware_precision"] == 0.9567

                          Split Precision
                   Random split    92.04%
March -> April time-aware split    95.67%


## 6. Limitations

This model only detects symptoms of search traffic drops in measured search fields. It remains blind to traffic from social media, direct links, or email. It cannot identify purely informational “zero-click” search intents.

The findings are observed, measured, and directional. They do not predict Google's algorithm, establish causality, or guarantee that a refresh will recover traffic. Every output requires human review.

## 7. Ranked recommendations

The action playbook maps probability to a ranked editorial action:

- Probability `> 0.65`: `IMMEDIATE_REFRESH` — reason `HIGH_VOL_LOW_CTR_PAGE_1`.
- Probability `0.45–0.65`: `METADATA_TWEAK` — reason `BORDERLINE_POSITION_DROP`.
- Probability `< 0.45`: `NO_ACTION` — reason `HEALTHY_METRICS`.

The queue is for human review only. It never publishes automatically.

In [6]:
def action_for_probability(probability):
    if probability > 0.65:
        return ("IMMEDIATE_REFRESH", "HIGH_VOL_LOW_CTR_PAGE_1")
    if probability >= 0.45:
        return ("METADATA_TWEAK", "BORDERLINE_POSITION_DROP")
    return ("NO_ACTION", "HEALTHY_METRICS")

example_probabilities = [0.80, 0.65, 0.50, 0.45, 0.20]
playbook_check = pd.DataFrame([
    {"probability": probability, "action": action_for_probability(probability)[0], "reason": action_for_probability(probability)[1]}
    for probability in example_probabilities
])
print(playbook_check.to_string(index=False))
assert action_for_probability(0.66) == ("IMMEDIATE_REFRESH", "HIGH_VOL_LOW_CTR_PAGE_1")
assert action_for_probability(0.65) == ("METADATA_TWEAK", "BORDERLINE_POSITION_DROP")
assert action_for_probability(0.45) == ("METADATA_TWEAK", "BORDERLINE_POSITION_DROP")
assert action_for_probability(0.44) == ("NO_ACTION", "HEALTHY_METRICS")
print(f"Archived Week 7 example: {metrics['archived_playbook_review_count']} of {metrics['archived_playbook_scored_pages']} pages entered review.")

 probability            action                   reason
        0.80 IMMEDIATE_REFRESH  HIGH_VOL_LOW_CTR_PAGE_1
        0.65    METADATA_TWEAK BORDERLINE_POSITION_DROP
        0.50    METADATA_TWEAK BORDERLINE_POSITION_DROP
        0.45    METADATA_TWEAK BORDERLINE_POSITION_DROP
        0.20         NO_ACTION          HEALTHY_METRICS
Archived Week 7 example: 109 of 5000 pages entered review.


## 8. Artifacts and reproducibility

The paper-ready metrics receipt is `work/outputs/capstone_metrics.json`. The warehouse is gated, so a fresh rerun requires Hugging Face access and a read-only `HF_TOKEN` supplied through the runtime environment or Colab Secrets. Never commit tokens, warehouse data, raw queries, client names, or URLs.

The deployed paper is `docs/index.html`; enable GitHub Pages from the `docs/` folder and record the resulting HTTPS URL in `submission/paper_url.txt`.

In [7]:
print("Receipt exists:", metrics_path.exists())
print("Paper entry point:", repo_root / "docs" / "index.html")
print("No dataset artifact is created by this notebook.")

Receipt exists: True
Paper entry point: /Users/ibrahimyaser/Documents/CS Study/flyrank-ibrahim/docs/index.html
No dataset artifact is created by this notebook.


## ML-12 closing

### Five-minute demo outline

1. State the editorial decision: prioritize pages showing search-decay symptoms.
2. Show the March-to-June time-aware split and the leakage exclusions.
3. Compare the fixed heuristic with the archived Random Forest metrics.
4. Walk through the three probability bands and one human-review rule.
5. Close with the limitations: directional symptoms, not causal recovery or automated publishing.

### Social-post cut

I built a content-retention decision-support workflow that ranks pages for editorial review using historical search visibility metrics. On the archived June comparison, the Random Forest measured 99.97% precision versus 12.89% for a rigid Week 4 rule, while the paper keeps the claim directional and human-reviewed.

### Employer-facing summary

I built a reproducible Content Opportunity Scoring capstone around a time-aware March-to-June evaluation. It combines a transparent baseline, a Random Forest classifier, leakage checks, and a ranked action playbook for editors. The observed result supports prioritization research, while the documented limits prevent causal or automated-publishing claims.

## Self-check

- [x] Question, data, methodology, results, limitations, recommendations, artifacts, and ML-12 closing are filled.
- [x] The metrics receipt is committed and contains no dataset rows.
- [x] The notebook uses explicit observed/directional/decision-support language.
- [x] The final-month test window is described as June 2026 `_sample`.
- [x] The feature discrepancy between the requested contract and archived `w05` execution is disclosed.